In [1]:
import glob
from math import e

import os

import warnings
import imageio.v2 as imageio

import kornia
import numpy as np
import torch
import torch.nn.functional as F

import torch.utils.data
import torchvision.transforms as transforms


from torch import nn

from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import RandomAffine, ToPILImage
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim
from module import Res_CNN, Restormer, ResViT

warnings.filterwarnings("ignore")


def PSNR(real, fake):
    x, y = np.where(real != -1)  # Exclude background
    mse = np.mean(((fake[x][y] + 1) / 2.0 - (real[x][y] + 1) / 2.0) ** 2)
    if mse < 1.0e-10:
        return 100
    else:
        PIXEL_MAX = 1
        return 20 * np.log10(PIXEL_MAX / np.sqrt(mse))


def smooothing_loss(y_pred):
    dy = torch.abs(y_pred[:, :, 1:, :] - y_pred[:, :, :-1, :])
    dx = torch.abs(y_pred[:, :, :, 1:] - y_pred[:, :, :, :-1])

    dx = dx * dx
    dy = dy * dy
    d = torch.mean(dx) + torch.mean(dy)
    grad = d
    return d


class Sobelxy(nn.Module):
    def __init__(self):
        super(Sobelxy, self).__init__()
        kernelx = [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]
        kernely = [[1, 2, 1], [0, 0, 0], [-1, -2, -1]]
        kernelx = torch.FloatTensor(kernelx).unsqueeze(0).unsqueeze(0)
        kernely = torch.FloatTensor(kernely).unsqueeze(0).unsqueeze(0)
        self.weightx = nn.Parameter(data=kernelx, requires_grad=False).cuda()
        self.weighty = nn.Parameter(data=kernely, requires_grad=False).cuda()

    def forward(self, x):
        sobelx = F.conv2d(x, self.weightx, padding=1)
        sobely = F.conv2d(x, self.weighty, padding=1)
        return torch.abs(sobelx) + torch.abs(sobely)


class GradLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.sobelconv = Sobelxy()

    def forward(self, x, y):
        x_grad = self.sobelconv(x)
        y_grad = self.sobelconv(y)
        grad_loss = F.l1_loss(x_grad, y_grad)
        return grad_loss


def normalize_matrix(matrix):
    # 找到矩阵的最小值和最大值
    min_val = np.min(matrix)
    max_val = np.max(matrix)

    # 归一化到[-1, 1]
    normalized_matrix = 2 * (matrix - min_val) / (max_val - min_val) - 1

    return normalized_matrix


def CreateDatasetSynthesis(phase, input_path, contrast1="t1", contrast2="t2"):
    root = os.path.join(input_path, phase)

    dataset = ImageDataset1(root, contrast1, contrast2)
    return dataset


class ToTensor:
    def __call__(self, tensor):
        tensor = np.expand_dims(tensor, 0)
        return torch.from_numpy(tensor)


class Resize:
    def __init__(self, size_tuple, use_cv=True):
        self.size_tuple = size_tuple
        self.use_cv = use_cv


class ImageDataset1(Dataset):
    def __init__(self, root, contrast1, contrast2):
        self.files_A = sorted(glob.glob(os.path.join(root, contrast1 + "_npy/*")))
        self.files_B = sorted(glob.glob(os.path.join(root, contrast2 + "_npy/*")))

        transforms_2 = [
            ToPILImage(),
            RandomAffine(
                degrees=1, translate=[0.02, 0.02], scale=[0.98, 1.02], fill=-1
            ),
            ToTensor(),
            # Resize(size_tuple=(256, 256)),
        ]
        self.transform2 = transforms.Compose(transforms_2)

    def __getitem__(self, index):
        seed = np.random.randint(2147483647)  # make a seed with numpy generator
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        item_A = self.transform2(
            np.load(self.files_A[index % len(self.files_A)]).astype(np.float32)
        )

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        item_B = self.transform2(
            np.load(self.files_B[index % len(self.files_B)]).astype(np.float32)
        )

        return item_A, item_B

    def __len__(self):
        return max(len(self.files_A), len(self.files_B))


def pair(t):
    return t if isinstance(t, tuple) else (t, t)


dataset = CreateDatasetSynthesis(
    phase="train",
    input_path="/root/workspace/datasets/I2I/IXI/",
    contrast1="t1",
    contrast2="t2",
)
data_loader = DataLoader(
    dataset, batch_size=1, shuffle=True, num_workers=4, drop_last=True
)
val_dataset = CreateDatasetSynthesis(
    phase="val",
    input_path="/root/workspace/datasets/I2I/IXI/",
    contrast1="t1",
    contrast2="t2",
)
val_data = DataLoader(
    val_dataset, batch_size=1, shuffle=False, num_workers=4, drop_last=True
)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
lr = 5e-5
weight_decay = 0
optim_step = 10
optim_gamma = 0.7

# model = Generator(1, 1)
model = ResViT()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=optim_step, gamma=optim_gamma
)
Loss_MSE = nn.MSELoss()
Loss_L1 = nn.L1Loss()
Loss_SSIM = kornia.losses.SSIMLoss(11, reduction="mean")
Loss_grad = GradLoss()
model.to(device)
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    loss_epoch = 0
    train_iter = len(data_loader)
    for x_1, x_2 in tqdm(data_loader):
        real_a, real_b = x_1.to(device), x_2.to(device)
        fake_b = model(real_a)

        loss = (
            5 * Loss_SSIM(real_b, fake_b)
            + Loss_L1(real_b, fake_b)
            + 0.1 * Loss_grad(real_b, fake_b)
        )

        loss_epoch += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("loss =", loss_epoch / len(data_loader))

    with torch.no_grad():
        model.eval()
        psnr_avg = 0
        ssim_avg = 0
        for i, ab_pair in enumerate(tqdm(val_data)):
            real_a, real_b = ab_pair
            real_a, real_b = real_a.to(device), real_b.to(device)
            fake_b = model(real_a)

            real_a = real_a[0, 0, :, :].squeeze().cpu().numpy()
            real_b = real_b[0, 0, :, :].squeeze().cpu().numpy()
            fake_b = fake_b[0, 0, :, :].squeeze().cpu().numpy()

            psnr_avg += PSNR(real_b, fake_b)
            ssim_avg += ssim(real_b, fake_b, data_range=2)

            real_a = (
                (real_a - real_a.min()) * (1 / (real_a.max() - real_a.min()) * 255)
            ).astype("uint8")
            real_b = (
                (real_b - real_b.min()) * (1 / (real_b.max() - real_b.min()) * 255)
            ).astype("uint8")
            fake_b = (
                (fake_b - fake_b.min()) * (1 / (fake_b.max() - fake_b.min()) * 255)
            ).astype("uint8")
            real_fake_b = np.hstack((real_a, real_b, fake_b))
            imageio.imwrite(
                "/root/workspace/DSLFuse/log/imsave2/" + "real_fake_" + str(i) + ".png",
                real_fake_b,
            )

        print("psnr =", psnr_avg / len(val_data))
        print("ssim =", ssim_avg / len(val_data))
        print("====================", epoch + 1, "====================")

    if scheduler is not None:
        scheduler.step()

100%|██████████| 2400/2400 [04:05<00:00,  9.78it/s]


loss = 0.7646261952072382


100%|██████████| 800/800 [01:54<00:00,  6.96it/s]


psnr = 51.21615733173192
ssim = 0.7866270852633304
==================== 1 ====================


100%|██████████| 2400/2400 [04:02<00:00,  9.88it/s]


...

100%|██████████| 2400/2400 [07:50<00:00,  5.10it/s]


loss = 0.362727428060025


100%|██████████| 800/800 [02:16<00:00,  5.88it/s]


psnr = 51.9019926168231
ssim = 0.806300000767637
==================== 23 ====================


 92%|█████████▏| 2214/2400 [07:13<00:36,  5.08it/s]